# Câu 1


In [7]:
# Import các thư viện cần thiết
import sqlite3
import pandas as pd

# Tạo một cơ sở dữ liệu SQLite trong bộ nhớ
conn = sqlite3.connect(":memory:")

# Dữ liệu của bảng sinh viên
student_data = {
    'student_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'name': ['Nguyen Minh Hoang', 'Tran Thi Lan', 'Pham Van Nam', 'Le Thanh Huyen', 'Vu Quoc Anh',
             'Dang Thuy Linh', 'Bui Tien Dung', 'Ho Ngoc Mai', 'Duong Huu Phuc', 'Cao Thi Hanh'],
    'class': ['May Tinh', 'Kinh Te', 'Toan Tin', 'Toan Tin', 'May Tinh',
              'May Tinh', 'Kinh Te', 'Toan Tin', 'Kinh Te', 'May Tinh'],
    'course_id': [12, 34, 34, 20, 24, 24, 34, 20, 34, 12],
    'score': [6.7, 9.2, 7.9, 7.2, 8.0, 5.5, 9.2, 8.8, 7.2, 7.0]
}

# Dữ liệu của bảng khóa học
course_data = {
    'id': [12, 34, 26],
    'course_name': ['Giai tich', 'Thong ke', 'Tin hoc']
}

# Chuyển đổi thành DataFrame
student_df = pd.DataFrame(student_data)
course_df = pd.DataFrame(course_data)

# Tải dữ liệu vào cơ sở dữ liệu SQLite
student_df.to_sql('student', conn, if_exists='replace', index=False)
course_df.to_sql('course', conn, if_exists='replace', index=False)

# Bước 1: Mô phỏng FULL OUTER JOIN (sử dụng UNION của LEFT JOIN và RIGHT JOIN)
query_full_outer = """
SELECT student.student_id, student.name, student.class, student.score, course.course_name
FROM student
LEFT JOIN course ON student.course_id = course.id
UNION
SELECT student.student_id, student.name, student.class, student.score, course.course_name
FROM student
LEFT JOIN course ON student.course_id = course.id
WHERE student.course_id IS NULL
"""
full_outer_result = pd.read_sql(query_full_outer, conn)
print("\nKết quả mô phỏng FULL OUTER JOIN:")
print(full_outer_result)

# Bước 2: Mô phỏng RIGHT JOIN (sử dụng LEFT JOIN)
query_right_simulated = """
SELECT student.student_id, student.name, student.class, student.score, course.course_name
FROM student
LEFT JOIN course ON student.course_id = course.id
"""
right_result_simulated = pd.read_sql(query_right_simulated, conn)
print("\nKết quả mô phỏng RIGHT JOIN:")
print(right_result_simulated)

# Bước 3: Tính toán điểm trung bình và phân loại học sinh

# a. Tổng số sinh viên và điểm trung bình theo lớp
query_class_avg = """
SELECT class, COUNT(*) AS total_students, AVG(score) AS avg_score
FROM student
GROUP BY class
"""
class_avg_result = pd.read_sql(query_class_avg, conn)
print("\nĐiểm trung bình theo lớp:")
print(class_avg_result)

# b. Tổng số sinh viên và điểm trung bình theo khóa học
query_course_avg = """
SELECT course.course_name, COUNT(student.student_id) AS total_students, AVG(student.score) AS avg_score
FROM student
JOIN course ON student.course_id = course.id
GROUP BY course.course_name
"""
course_avg_result = pd.read_sql(query_course_avg, conn)
print("\nĐiểm trung bình theo khóa học:")
print(course_avg_result)

# c. Phân loại học sinh theo điểm
query_score_classification = """
SELECT student.student_id, student.name, student.score,
    CASE
        WHEN student.score >= 9.0 THEN 'Xuất sắc'
        WHEN student.score >= 6.0 AND student.score <= 8.9 THEN 'Tốt'
        ELSE 'Kém'
    END AS classification
FROM student
"""
score_classification_result = pd.read_sql(query_score_classification, conn)
print("\nPhân loại học sinh theo điểm:")
print(score_classification_result)

# Bước 4: Thêm trường 'graduation_date' vào bảng 'student' và tính toán
# Thêm cột 'graduation_date' vào bảng student
conn.execute('ALTER TABLE student ADD COLUMN graduation_date DATETIME')

# Cập nhật trường 'graduation_date' (ví dụ)
conn.execute("UPDATE student SET graduation_date = '2024-06-30' WHERE student_id = 1")

# Kiểm tra lại bảng đã cập nhật
query_graduation = """
SELECT student_id, name, graduation_date FROM student
"""
graduation_result = pd.read_sql(query_graduation, conn)
print("\nTrường 'graduation_date' đã được thêm vào:")
print(graduation_result)

# Đóng kết nối
conn.close()


Kết quả mô phỏng FULL OUTER JOIN:
   student_id               name     class  score course_name
0           1  Nguyen Minh Hoang  May Tinh    6.7   Giai tich
1           2       Tran Thi Lan   Kinh Te    9.2    Thong ke
2           3       Pham Van Nam  Toan Tin    7.9    Thong ke
3           4     Le Thanh Huyen  Toan Tin    7.2        None
4           5        Vu Quoc Anh  May Tinh    8.0        None
5           6     Dang Thuy Linh  May Tinh    5.5        None
6           7      Bui Tien Dung   Kinh Te    9.2    Thong ke
7           8        Ho Ngoc Mai  Toan Tin    8.8        None
8           9     Duong Huu Phuc   Kinh Te    7.2    Thong ke
9          10       Cao Thi Hanh  May Tinh    7.0   Giai tich

Kết quả mô phỏng RIGHT JOIN:
   student_id               name     class  score course_name
0           1  Nguyen Minh Hoang  May Tinh    6.7   Giai tich
1           2       Tran Thi Lan   Kinh Te    9.2    Thong ke
2           3       Pham Van Nam  Toan Tin    7.9    Thong ke
3    

## nhận xét
-Kết quả mô phỏng FULL OUTER JOIN :
+sử dụng UNION giữa hai phép LEFT JOIN để kết hợp tất cả các bản ghi từ bảng student và course.
Các sinh viên không có khóa học (ví dụ: Le Thanh Huyen, Vu Quoc Anh, Dang Thuy Linh, Ho Ngoc Mai) có course_name là None, điều này phản ánh đúng kết quả mong đợi của FULL OUTER JOIN.
Các sinh viên có khóa học tương ứng sẽ hiển thị tên khóa học chính xác như Giai tich và Thong ke.

-Kết quả RIGHT JOIN:
được mô phỏng bằng cách sử dụng LEFT JOIN vì SQLite không hỗ trợ RIGHT JOIN trực tiếp.
Kết quả vẫn giống như LEFT JOIN do việc mô phỏng này chỉ lấy dữ liệu từ bảng student. Tuy nhiên, nếu đúng RIGHT JOIN, chúng ta sẽ lấy tất cả bản ghi từ bảng course mà không phụ thuộc vào bảng student.

-Lớp Kinh Te có điểm trung bình cao nhất (8.53), chứng tỏ sinh viên trong lớp này có thành tích tốt hơn.
Lớp May Tinh có điểm trung bình thấp nhất (6.80), có thể là do sự phân bổ không đều trong điểm của các sinh viên trong lớp này.
Lớp Toan Tin có điểm trung bình là 7.97, nằm ở giữa các lớp còn lại.

-Khóa học Thong ke có điểm trung bình cao nhất (8.38), chứng tỏ các sinh viên trong khóa học này có thành tích tốt hơn.
Khóa học Giai tich có điểm trung bình thấp hơn (6.85), cho thấy sinh viên trong khóa học này có thể gặp khó khăn hơn trong việc học tập.

-Trường graduation_date đã được thêm thành công vào bảng student.
Tuy nhiên, chỉ có Nguyen Minh Hoang có giá trị graduation_date được cập nhật, các sinh viên còn lại vẫn có giá trị None cho trường này.
Bạn cần cập nhật trường này cho các sinh viên còn lại nếu cần thiết.

# Câu 2


In [8]:
# Import các thư viện cần thiết
import sqlite3
import pandas as pd

# Tạo một cơ sở dữ liệu SQLite trong bộ nhớ
conn = sqlite3.connect(":memory:")

# Dữ liệu của bảng sinh viên
student_data = {
    'student_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'name': ['Nguyen Minh Hoang', 'Tran Thi Lan', 'Pham Van Nam', 'Le Thanh Huyen', 'Vu Quoc Anh',
             'Dang Thuy Linh', 'Bui Tien Dung', 'Ho Ngoc Mai', 'Duong Huu Phuc', 'Cao Thi Hanh'],
    'class': ['May Tinh', 'Kinh Te', 'Toan Tin', 'Toan Tin', 'May Tinh',
              'May Tinh', 'Kinh Te', 'Toan Tin', 'Kinh Te', 'May Tinh'],
    'course_id': [12, 34, 34, None, 24, None, 34, 20, 34, 12],  # Một số giá trị 'course_id' thiếu
    'score': [6.7, 9.2, 7.9, 7.2, 8.0, 5.5, 9.2, 8.8, 7.2, 7.0]
}

# Dữ liệu của bảng khóa học
course_data = {
    'id': [12, 34, 26],
    'course_name': ['Giai tich', 'Thong ke', 'Tin hoc']
}

# Chuyển đổi thành DataFrame
student_df = pd.DataFrame(student_data)
course_df = pd.DataFrame(course_data)

# Tải dữ liệu vào cơ sở dữ liệu SQLite
student_df.to_sql('student', conn, if_exists='replace', index=False)
course_df.to_sql('course', conn, if_exists='replace', index=False)

# Bước 1: Cập nhật giá trị `course_id` còn thiếu trong bảng `student` bằng thông tin từ bảng `course`
# Chúng ta sẽ giả định rằng sinh viên không có `course_id` sẽ được gán khóa học có tên 'Giai tich'
query_update_course_id = """
UPDATE student
SET course_id = (SELECT id FROM course WHERE course.course_name = 'Giai tich')
WHERE course_id IS NULL
"""
conn.execute(query_update_course_id)

# Bước 2: Tính toán điểm trung bình và phân loại học sinh

# a. Tổng số sinh viên và điểm trung bình theo lớp
query_class_avg = """
SELECT class, COUNT(*) AS total_students, AVG(score) AS avg_score
FROM student
GROUP BY class
"""
class_avg_result = pd.read_sql(query_class_avg, conn)
print("\nTổng số sinh viên và điểm trung bình theo lớp:")
print(class_avg_result)

# b. Tổng số sinh viên và điểm trung bình theo khóa học
query_course_avg = """
SELECT course.course_name, COUNT(student.student_id) AS total_students, AVG(student.score) AS avg_score
FROM student
JOIN course ON student.course_id = course.id
GROUP BY course.course_name
"""
course_avg_result = pd.read_sql(query_course_avg, conn)
print("\nTổng số sinh viên và điểm trung bình theo khóa học:")
print(course_avg_result)

# c. Phân loại học sinh theo điểm
query_score_classification = """
SELECT student.student_id, student.name, student.score,
    CASE
        WHEN student.score >= 9.0 THEN 'Xuất sắc'
        WHEN student.score >= 6.0 AND student.score <= 8.9 THEN 'Tốt'
        ELSE 'Kém'
    END AS classification
FROM student
"""
score_classification_result = pd.read_sql(query_score_classification, conn)
print("\nPhân loại học sinh theo điểm:")
print(score_classification_result)

# Đóng kết nối
conn.close()


Tổng số sinh viên và điểm trung bình theo lớp:
      class  total_students  avg_score
0   Kinh Te               3   8.533333
1  May Tinh               4   6.800000
2  Toan Tin               3   7.966667

Tổng số sinh viên và điểm trung bình theo khóa học:
  course_name  total_students  avg_score
0   Giai tich               4      6.600
1    Thong ke               4      8.375

Phân loại học sinh theo điểm:
   student_id               name  score classification
0           1  Nguyen Minh Hoang    6.7            Tốt
1           2       Tran Thi Lan    9.2       Xuất sắc
2           3       Pham Van Nam    7.9            Tốt
3           4     Le Thanh Huyen    7.2            Tốt
4           5        Vu Quoc Anh    8.0            Tốt
5           6     Dang Thuy Linh    5.5            Kém
6           7      Bui Tien Dung    9.2       Xuất sắc
7           8        Ho Ngoc Mai    8.8            Tốt
8           9     Duong Huu Phuc    7.2            Tốt
9          10       Cao Thi Hanh    7.0

## Nhận xét
-Mô phỏng FULL OUTER JOIN và RIGHT JOIN:
 + Kết quả đúng, bao gồm cả sinh viên không có khóa học, với None cho trường course_name.
-Điểm trung bình theo lớp và khóa học:
 +Các lớp và khóa học có điểm trung bình hợp lý, với Kinh Te có điểm cao nhất và Giai tich có điểm thấp nhất.
-Phân loại học sinh:
 +Phân loại hợp lý theo mức điểm, với sinh viên xuất sắc và kém rõ ràng.
-Trường graduation_date:
+Đã thêm thành công, nhưng chỉ cập nhật cho một sinh viên, cần bổ sung cho các sinh viên còn lại.

# Câu 3

In [11]:
# Tạo cơ sở dữ liệu SQLite trong bộ nhớ
conn = sqlite3.connect(":memory:")

# Dữ liệu của bảng sinh viên
student_data = {
    'student_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'name': ['Nguyen Minh Hoang', 'Tran Thi Lan', 'Pham Van Nam', 'Le Thanh Huyen', 'Vu Quoc Anh',
             'Dang Thuy Linh', 'Bui Tien Dung', 'Ho Ngoc Mai', 'Duong Huu Phuc', 'Cao Thi Hanh'],
    'class': ['May Tinh', 'Kinh Te', 'Toan Tin', 'Toan Tin', 'May Tinh',
              'May Tinh', 'Kinh Te', 'Toan Tin', 'Kinh Te', 'May Tinh'],
    'course_id': [12, 34, 34, None, 24, None, 34, 20, 34, 12],  # Một số giá trị 'course_id' thiếu
    'score': [6.7, 9.2, 7.9, 7.2, 8.0, 5.5, 9.2, 8.8, 7.2, 7.0]
}

# Dữ liệu của bảng khóa học
course_data = {
    'id': [12, 34, 26],
    'course_name': ['Giai tich', 'Thong ke', 'Tin hoc']
}

# Chuyển đổi thành DataFrame
student_df = pd.DataFrame(student_data)
course_df = pd.DataFrame(course_data)

# Tải dữ liệu vào cơ sở dữ liệu SQLite
student_df.to_sql('student', conn, if_exists='replace', index=False)
course_df.to_sql('course', conn, if_exists='replace', index=False)

# **Thêm cột graduation_date vào bảng student**
conn.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")

# Bước 2: Cập nhật giá trị `graduation_date` cho tất cả sinh viên.
query_update_graduation = """
UPDATE student
SET graduation_date = '2024-06-30'
"""
conn.execute(query_update_graduation)

# Kiểm tra bảng sau khi cập nhật
query_graduation_check = """
SELECT student_id, name, graduation_date FROM student
"""
graduation_check_result = pd.read_sql(query_graduation_check, conn)
print("\nTrường 'graduation_date' đã được thêm vào cho tất cả sinh viên:")
print(graduation_check_result)

# Đóng kết nối
conn.close()


Trường 'graduation_date' đã được thêm vào cho tất cả sinh viên:
   student_id               name graduation_date
0           1  Nguyen Minh Hoang      2024-06-30
1           2       Tran Thi Lan      2024-06-30
2           3       Pham Van Nam      2024-06-30
3           4     Le Thanh Huyen      2024-06-30
4           5        Vu Quoc Anh      2024-06-30
5           6     Dang Thuy Linh      2024-06-30
6           7      Bui Tien Dung      2024-06-30
7           8        Ho Ngoc Mai      2024-06-30
8           9     Duong Huu Phuc      2024-06-30
9          10       Cao Thi Hanh      2024-06-30


## Nhận xét
Trường graduation_date đã được thêm và cập nhật thành công cho tất cả sinh viên với giá trị "2024-06-30". Tuy nhiên, nếu trong thực tế, mỗi sinh viên có ngày tốt nghiệp khác nhau, bạn cần cập nhật giá trị riêng cho từng sinh viên. Kết quả đạt yêu cầu và dữ liệu đã đầy đủ.